# Predictive maintenance — datasetul din S3

**Scop.** Construiește tabelul de ferestre pentru detectorul de anomalii, citind direct din
bucketul corpusului. Fiecare sesiune trece prin `cncdt_ml.mujoco_corpus.build_windows`, deci
caracteristicile sunt identice cu cele din Lambda.

**Intrare.** `s3://cncdt-dev-training-data-864981730944/training-data/mujoco/`.
**Ieșire.** `s3://cncdt-dev-datalake-864981730944/features/predictive-maintenance/v1/windows.parquet`.

Rulează pe un kernel cu 8 GiB (`ml.m5.large`): cea mai mare sesiune are ~350k eșantioane.



## 1. Cod, importuri și căi


In [ ]:
from pathlib import Path
import json, sys, tarfile

import boto3
import pandas as pd

CORPUS = "cncdt-dev-training-data-864981730944"
PREFIX = "training-data/mujoco/"
DL = "cncdt-dev-datalake-864981730944"
CODE = Path.home() / "cnc-code"

s3 = boto3.client("s3")
bundle = Path("/tmp/cncdt-code.tgz")
s3.download_file(DL, "notebooks/studio/code/cncdt-code.tgz", str(bundle))
CODE.mkdir(exist_ok=True)
tarfile.open(bundle).extractall(CODE)
sys.path.insert(0, str(CODE / "ml/src"))

from cncdt_ml.mujoco_corpus import build_windows

print("cod:", CODE)



## 2. Inventarul din S3

`manifest.json` al fiecărei rulări dă sesiunile, dimensiunea și starea de uzură fără să citească eșantioane.


In [ ]:
objects = [o for page in s3.get_paginator("list_objects_v2").paginate(Bucket=CORPUS, Prefix=PREFIX)
           for o in page.get("Contents", [])]
print(f"{len(objects)} obiecte, {sum(o['Size'] for o in objects) / 2**30:.2f} GiB")

runs, sessions = [], []
for key in sorted(o["Key"] for o in objects if o["Key"].endswith("/manifest.json")):
    manifest = json.loads(s3.get_object(Bucket=CORPUS, Key=key)["Body"].read())
    rulare = key.split("/")[-2]
    runs.append({"rulare": rulare, "data": key.split("/")[-3],
                 "sesiuni": manifest["totals"]["sessions"],
                 "esantioane": manifest["totals"]["samples"],
                 "GiB": round(manifest["totals"]["bytes"] / 2**30, 2),
                 "sanatate_min": manifest["totals"]["health_min"],
                 "esecuri": manifest["totals"]["failures"]})
    for item in manifest["sessions"]:
        sessions.append({"rulare": rulare, "nume": item["name"], "prefix": item["prefix"],
                         "familie": item["family"], "esantioane": item["samples"],
                         "defecte": item["defects"], "acoperire": item["coverage"]})

display(pd.DataFrame(runs))


# Corpusul are și rulări vechi, de zeci de milioane de eșantioane. Pentru modelul curent
# ne interesează rularea sănătoasă de referință și rularea nouă de uzură; restul ar lungi
# descărcarea fără să schimbe antrenarea.
RULARI = {"run-20261018T230709", "run-20261024T230425"}
alese = [s for s in sessions if s["rulare"] in RULARI]
if alese:
    sessions = alese
    print(f"procesez {len(sessions)} sesiuni din {sorted(RULARI)}")
else:
    print("rulările țintă lipsesc; procesez tot corpusul")


## 3. Ferestrele, sesiune cu sesiune

Fiecare sesiune se descarcă temporar în instanță (`session.json` și `samples.jsonl`, cel mult
~170 MB), se transformă în ferestre de 60 s, apoi directorul temporar dispare.



In [ ]:
import tempfile

rows = []
for i, session in enumerate(sessions, 1):
    with tempfile.TemporaryDirectory() as tmp:
        for name in ("session.json", "samples.jsonl"):
            s3.download_file(CORPUS, session["prefix"] + name, str(Path(tmp) / name))
        windows = build_windows(Path(tmp), window_s=60)
    for row in windows:
        row["rulare"] = session["rulare"]
        row["sesiune"] = session["nume"]
    rows.extend(windows)
    print(f"{i}/{len(sessions)} {session['nume']}: {len(windows)} ferestre", flush=True)

features = pd.DataFrame(rows)
print(features.shape)
features["twin_label"].value_counts()



## 4. Eticheta de mentenanță și salvarea

`degradat` se calculează din uzura sculei, frecarea articulațiilor și temperatură. Pragul de
0,50 este nivelul „worn” din `wear.py`: componenta cere planificare, nu doar observație.
Rulările scurte de 20 de cicluri la `rate 5` nu ajung la 0,75, iar un prag prea sus ar da zero
exemple degradate. Defectul de piesă rămâne etichetă separată, pentru modelul de viziune.


In [ ]:
# Nivelul „worn” din wear.py. Trebuie să fie același în pm-02, altfel eticheta de evaluare
# nu mai înseamnă același lucru în cele două notebook-uri.
DEGRADED_TOOL_WEAR = 0.50
DEGRADED_JOINT_FRICTION = 0.50
DEGRADED_TEMP_C = 78.0

# Lambda-ul cheamă modelul doar pe ferestre cu cel puțin 30 de eșantioane de așchiere
# (`MIN_CUTTING_SAMPLES`). Fără filtru, antrenarea ar vedea și ferestre de repaus, cu
# caracteristici zero, care nu ajung niciodată la endpoint.
MIN_CUTTING_SAMPLES = 30
features = features[features["cutting_samples"] >= MIN_CUTTING_SAMPLES].reset_index(drop=True)

features["degradat"] = ((features["tool_wear_index_max"] >= DEGRADED_TOOL_WEAR) |
                        (features["joint_friction_index_max"] >= DEGRADED_JOINT_FRICTION) |
                        (features["temp_c_max"] >= DEGRADED_TEMP_C))

display(features.groupby("rulare").agg(ferestre=("healthy", "size"),
                                       sanatoase=("healthy", "sum"),
                                       degradate=("degradat", "sum"),
                                       uzura_max=("tool_wear_index_max", "max"),
                                       frecare_max=("joint_friction_index_max", "max"),
                                       temp_max=("temp_c_max", "max")))

features.to_parquet("/tmp/pm-windows.parquet", index=False)
key = "features/predictive-maintenance/v1/windows.parquet"
s3.upload_file("/tmp/pm-windows.parquet", DL, key)
print("scris s3://" + DL + "/" + key)


## 5. Pasul următor

Deschide `pm-02-rcf.ipynb`. Antrenarea RCF folosește ferestrele sănătoase, iar evaluarea se face
pe o rulare degradată ținută deoparte.

